# 🚩 POC 3: Feature Flags — A/B Test ML Models

**What you'll learn:**
- Feature Flags = toggle features ON/OFF without redeploying
- A/B testing two ML models in production
- Kill switches for production incidents
- Percentage rollouts and target groups
- Harness Feature Flags SDK (Python)

**Real-world AI/ML use cases:**
| Use Case | How |
|----------|-----|
| A/B test new model | 50% users see model_v1, 50% see model_v2 |
| Kill switch | Disable ML predictions if model is broken |
| Gradual rollout | New model to 10% → 50% → 100% of users |
| Beta features | Show new UI only to beta testers |

In [ ]:
# Step 1: Build a Feature Flag system from scratch
# This simulates EXACTLY how the Harness FF SDK works internally

import hashlib, json, time
from typing import Any

class FeatureFlagClient:
    """
    Simulates Harness Feature Flags SDK.
    In production, install: pip install harness-featureflags
    """
    
    def __init__(self, sdk_key: str = "local-dev-key"):
        self.sdk_key = sdk_key
        self.flags = {}   # flag_key → {value, rules, kind}
        self.log = []     # audit trail
        print(f"🚩 Feature Flag Client initialized (key: {sdk_key[:10]}...)")
    
    def create_flag(self, key: str, default: Any, kind: str = "boolean",
                    description: str = "", rules: list = None):
        """Create a feature flag (done in Harness UI normally)"""
        self.flags[key] = {
            "value": default, "default": default,
            "kind": kind, "description": description,
            "rules": rules or []
        }
        print(f"  ✅ Created flag: {key} = {default} ({kind})")
    
    def evaluate(self, flag_key: str, target: dict, default: Any = None) -> Any:
        """Evaluate a flag for a specific target (user)"""
        flag = self.flags.get(flag_key)
        if not flag:
            return default
        
        # Check rules (priority: target group → percentage → default)
        for rule in flag.get("rules", []):
            # Rule: Target group match
            if "group" in rule and rule["group"] in target.get("groups", []):
                self._log(flag_key, target["id"], rule["serve"], "group_match")
                return rule["serve"]
            # Rule: Percentage rollout
            if "percentage" in rule:
                bucket = self._hash_bucket(target["id"], flag_key)
                if bucket < rule["percentage"]:
                    self._log(flag_key, target["id"], rule["serve"], f"pct_{rule['percentage']}")
                    return rule["serve"]
        
        self._log(flag_key, target["id"], flag["value"], "default")
        return flag["value"]
    
    def toggle(self, flag_key: str, value: Any):
        """Toggle flag value (simulates Harness UI toggle)"""
        if flag_key in self.flags:
            old = self.flags[flag_key]["value"]
            self.flags[flag_key]["value"] = value
            print(f"  🔄 {flag_key}: {old} → {value}")
    
    def _hash_bucket(self, user_id: str, flag_key: str) -> int:
        """Deterministic bucket (0-99) for percentage rollouts"""
        h = hashlib.md5(f"{user_id}:{flag_key}".encode()).hexdigest()
        return int(h[:8], 16) % 100
    
    def _log(self, flag, target, value, reason):
        self.log.append({"flag": flag, "target": target, "value": value, "reason": reason})

# Initialize client
ff = FeatureFlagClient("my-harness-sdk-key-abc123")
print("\n📋 Setting up flags...")

# Create flags for our ML platform
ff.create_flag("use_new_model", default=False, kind="boolean",
               description="Switch to new sentiment model v2",
               rules=[
                   {"group": "beta_testers", "serve": True},  # Beta users get new model
                   {"percentage": 20, "serve": True},          # 20% of everyone else
               ])

ff.create_flag("enable_predictions", default=True, kind="boolean",
               description="Kill switch for ML predictions")

ff.create_flag("model_version", default="v1", kind="string",
               description="Which model version to use",
               rules=[{"group": "enterprise", "serve": "v3_enterprise"}])

ff.create_flag("max_batch_size", default=10, kind="number",
               description="Max batch size for predictions")

In [ ]:
# Step 2: A/B Test Two ML Models Using Feature Flags

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

# Training data
texts = ["I love this", "Amazing product", "Great quality", "Best ever",
         "Terrible", "Worst experience", "Very bad", "Hate it"]
labels = [1, 1, 1, 1, 0, 0, 0, 0]

# Model V1: Logistic Regression (current production model)
model_v1 = Pipeline([("tfidf", TfidfVectorizer()), ("clf", LogisticRegression())])
model_v1.fit(texts, labels)

# Model V2: Random Forest (new experimental model)
model_v2 = Pipeline([("tfidf", TfidfVectorizer()), ("clf", RandomForestClassifier(n_estimators=10))])
model_v2.fit(texts, labels)

models = {"v1": model_v1, "v2": model_v2}

def predict_with_flag(text: str, user: dict) -> dict:
    """
    Make a prediction using feature flags to decide which model.
    This is the REAL PATTERN used in production ML systems.
    """
    # Check kill switch first
    if not ff.evaluate("enable_predictions", user, True):
        return {"error": "Predictions disabled (kill switch)", "user": user["id"]}
    
    # Decide which model to use
    use_new = ff.evaluate("use_new_model", user, False)
    model_key = "v2" if use_new else "v1"
    model = models[model_key]
    
    pred = model.predict([text])[0]
    prob = model.predict_proba([text])[0]
    
    return {
        "text": text,
        "sentiment": "positive" if pred == 1 else "negative",
        "confidence": round(float(max(prob)), 3),
        "model": model_key,
        "user": user["id"]
    }

# Simulate different users
users = [
    {"id": "user_alice", "groups": ["beta_testers"]},   # Beta → gets v2
    {"id": "user_bob", "groups": []},                     # Regular → might get v2 (20%)
    {"id": "user_charlie", "groups": []},                  # Regular
    {"id": "user_enterprise1", "groups": ["enterprise"]},  # Enterprise
]

print("🧪 A/B TEST: Model v1 vs v2")
print("=" * 60)
for user in users:
    result = predict_with_flag("I love this product", user)
    groups = user.get('groups', [])
    print(f"  {user['id']:<20} groups={str(groups):<20} → model={result.get('model', 'N/A')}, sentiment={result.get('sentiment', 'N/A')}")

In [ ]:
# Step 3: Kill Switch Demo
# Scenario: Your ML model starts giving bad predictions at 3 AM
# Instead of redeploying, just flip the kill switch!

print("💀 KILL SWITCH DEMO")
print("=" * 50)

test_user = {"id": "user_test", "groups": []}

# Normal operation
result = predict_with_flag("Great product", test_user)
print(f"  Before: {result}")

# OH NO! Model is giving bad predictions! FLIP THE KILL SWITCH!
print("\n  ⚠️ Model is broken! Activating kill switch...")
ff.toggle("enable_predictions", False)

result = predict_with_flag("Great product", test_user)
print(f"  After:  {result}")

# Fix deployed, re-enable
print("\n  🔧 Fix deployed! Re-enabling predictions...")
ff.toggle("enable_predictions", True)

result = predict_with_flag("Great product", test_user)
print(f"  Restored: {result}")

print("\n💡 Key insight: Zero downtime, zero redeployment, instant toggle!")

In [ ]:
# Step 4: Percentage Rollout Simulation
# Gradually roll out new model: 20% → 50% → 100%

def simulate_rollout(num_users=100):
    """Simulate how many users get the new model"""
    v1_count = 0
    v2_count = 0
    for i in range(num_users):
        user = {"id": f"user_{i}", "groups": []}
        use_new = ff.evaluate("use_new_model", user, False)
        if use_new:
            v2_count += 1
        else:
            v1_count += 1
    return v1_count, v2_count

print("📊 GRADUAL ROLLOUT SIMULATION (100 users)")
print("=" * 50)

# Current: 20% rollout (from flag rules)
v1, v2 = simulate_rollout()
print(f"  20% rollout: v1={v1} users, v2={v2} users")
print(f"  {'█' * (v2 // 2)}{'░' * (v1 // 2)} ({v2}% on new model)")

# Increase to 50%
ff.flags["use_new_model"]["rules"] = [{"percentage": 50, "serve": True}]
v1, v2 = simulate_rollout()
print(f"\n  50% rollout: v1={v1} users, v2={v2} users")
print(f"  {'█' * (v2 // 2)}{'░' * (v1 // 2)} ({v2}% on new model)")

# Full rollout: 100%
ff.flags["use_new_model"]["rules"] = [{"percentage": 100, "serve": True}]
v1, v2 = simulate_rollout()
print(f"\n  100% rollout: v1={v1} users, v2={v2} users")
print(f"  {'█' * (v2 // 2)}{'░' * (v1 // 2)} ({v2}% on new model)")

print("\n💡 In Harness UI, you just drag a slider to change percentage!")

In [ ]:
# Step 5: How to use the REAL Harness Feature Flags SDK
# (This is what you'd use in production)

real_sdk_code = '''
# ============================================
# PRODUCTION CODE: Harness Feature Flags SDK
# ============================================
# pip install harness-featureflags

from featureflags.client import CfClient
from featureflags.evaluations.auth_target import Target

# 1. Initialize client with your SDK key
client = CfClient("YOUR_SDK_KEY_FROM_HARNESS")  # From Project → FF → Environments

# 2. Define target (user)
target = Target(
    identifier="user_123",
    name="John Doe",
    attributes={"plan": "enterprise", "country": "IN"}
)

# 3. Evaluate flags
use_new_model = client.bool_variation("use_new_model", target, False)
model_version = client.string_variation("model_version", target, "v1")
batch_size = client.number_variation("max_batch_size", target, 10)

# 4. Use in your ML code
if use_new_model:
    prediction = model_v2.predict(text)
else:
    prediction = model_v1.predict(text)

# 5. Close client on shutdown
client.close()
'''

print("📝 PRODUCTION SDK USAGE:")
print(real_sdk_code)

## 🎯 POC 3 Summary

**Key Learnings:**
1. Feature Flags **decouple deployment from release** — deploy code, toggle later
2. **Kill switches** = instant disable without redeploying
3. **Percentage rollouts** = gradual, safe rollout (10% → 50% → 100%)
4. **Target groups** = different behavior for different users (beta, enterprise)
5. **A/B testing** = compare model v1 vs v2 in production with real users

➡️ **Next: POC 4 — Full Production Pipeline (CI + CD + FF + Policies)!**